In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

koleksi_lagu = [
    "Jenny - Matimuda: Semoga matimu matimuda Semoga matiku matimuda Hidup tak perlu terlalu lama Jika dosa yang berkuasa",
    "Dongker - Bertaruh Pada Api: Aku ingin terus hidup Berharap umurku panjang Tapi lihat ku tak tenang Selamanya",
    "Efek Rumah Kaca - Balerina: Menghimpun energi Mengambil posisi Menjejakkan kaki Meniti temali",
    "FSTVLST - Hujan Mata Pisau: Selangit penuh mendung memperingatkan Seribuan mata pisau terhujan Dan payung bajaku biarkan tak berkembang Ketakutan yang menenggelamkan",
    "Alkateri - Zaman: Semua layu dan takkan pernah berarti Tiba waktunya semua dan akan terjadi Jadi katalis tak bisa hindari lagi Temui kekal dengan rasa sesal",
    "Float - Sementara: Percayalah hati lebih dari ini Pernah kita lalui Takkan lagi kita mesti jauh melangkah Nikmatilah lara",
    "Crayon Case - Gravits: Mengejar bayangan kilau yang kau tinggalkan Sebagai memori yang takkan pergi Mimpi kan berganti dan takkan pernah berhenti Kan ku katakan sampai jumpa di lain hari",
    "Murphy Radio - Penghujung Cerita: Dan sampailah kita pada akhirnya Di penghujung cerita Dan semua kisah terlukis indah 'Kan menjadi kenangan",
    "Rumahsakit - Panasea: Walau tak terucap, akan terus terdengar Nyanyian dalam hatimu Dalam angan dan nafasku Selamanya",
    "The Jeblogs - Sambutlah: Mungkin kita sampai Mungkin saja tidak Tugas kita hanyalah berjalan, oh"
]

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(koleksi_lagu)

skenario_uji = [
    {
        "query": "hidup",
        "relevan": {0, 1}
    },
    {
        "query": "api",
        "relevan": {1}
    },
    {
        "query": "kenangan cerita memori",
        "relevan": {6, 7}
    }
]

def precision_at_k(hasil, relevan, k):
    return sum(1 for d in hasil[:k] if d in relevan) / k

def recall(hasil, relevan):
    cocok = sum(1 for d in hasil if d in relevan)
    return cocok / len(relevan)

def f1(p, r):
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)

def average_precision(hasil, relevan):
    hit, total = 0, 0.0
    for i, d in enumerate(hasil, start=1):
        if d in relevan:
            hit += 1
            total += hit / i
    return total / len(relevan)

total_ap = 0.0

print("=== EVALUASI MESIN PENCARI LAGU ===")
for i, skenario in enumerate(skenario_uji, 1):
    query = skenario["query"]
    relevan = skenario["relevan"]
    
    query_vector = vectorizer.transform([query])
    skor = cosine_similarity(query_vector, X)
    hasil_ranking = sorted(enumerate(skor[0]), key=lambda x: x[1], reverse=True)
    hasil_indeks = [idx for idx, score in hasil_ranking if score > 0]
    
    while len(hasil_indeks) < 5:
        hasil_indeks.append(-1)

    p = precision_at_k(hasil_indeks, relevan, 5)
    r = recall(hasil_indeks, relevan)
    f = f1(p, r)
    ap = average_precision(hasil_indeks, relevan)
    
    total_ap += ap
    
    print(f"\nKueri {i}: '{query}'")
    print(f"Ground Truth : {relevan}")
    print(f"Hasil Sistem : {hasil_indeks[:5]} (Top 5)")
    print("-" * 30)
    print(f"Precision@5 : {p:.4f}")
    print(f"Recall      : {r:.4f}")
    print(f"F1-Score    : {f:.4f}")
    print(f"AP          : {ap:.4f}")

map_score = total_ap / len(skenario_uji)
print("\n" + "=" * 35)
print(f"NILAI MAP SISTEM: {map_score:.4f}")
print("=" * 35)

=== EVALUASI MESIN PENCARI LAGU ===

Kueri 1: 'hidup'
Ground Truth : {0, 1}
Hasil Sistem : [1, 0, -1, -1, -1] (Top 5)
------------------------------
Precision@5 : 0.4000
Recall      : 1.0000
F1-Score    : 0.5714
AP          : 1.0000

Kueri 2: 'api'
Ground Truth : {1}
Hasil Sistem : [1, -1, -1, -1, -1] (Top 5)
------------------------------
Precision@5 : 0.2000
Recall      : 1.0000
F1-Score    : 0.3333
AP          : 1.0000

Kueri 3: 'kenangan cerita memori'
Ground Truth : {6, 7}
Hasil Sistem : [7, 6, -1, -1, -1] (Top 5)
------------------------------
Precision@5 : 0.4000
Recall      : 1.0000
F1-Score    : 0.5714
AP          : 1.0000

NILAI MAP SISTEM: 1.0000
